In [1]:
def get_tc_data(days):

    # Check if required modules are installed, install if missing
    import sys
    import subprocess
    import importlib

    required_modules = ["pytz", "pandas"]
    for mod in required_modules:
        if importlib.util.find_spec(mod) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", mod])

    import os, urllib3, urllib.parse
    import pandas as pd
    from datetime import datetime, timedelta
    import pytz

    # --- Hardcoded settings ---
    LIB_PARENT = r"C:\Users\jaskew\Documents\project_repository\notebooks"
    OWNERS = ["HTOC Org"]
    VERIFY_SSL = False

    # --- sys.path injection
    if LIB_PARENT not in sys.path:
        sys.path.insert(0, LIB_PARENT)

    # Clear stale modules
    for m in [
        "IsolatedTCPythonLibrary",
        "IsolatedTCPythonLibrary.ThreatConnect",
        "IsolatedTCPythonLibrary.Owners",
        "IsolatedTCPythonLibrary.RequestObject",
        "IsolatedTCPythonLibrary.utils",
        "IsolatedTCPythonLibrary.utils.config_loader",
        "utils",
    ]:
        sys.modules.pop(m, None)

    # Imports
    from IsolatedTCPythonLibrary.ThreatConnect import ThreatConnect
    # Removed: Owners (unused)
    from IsolatedTCPythonLibrary.RequestObject import RequestObject
    from IsolatedTCPythonLibrary.utils.config_loader import load_config

    # Config
    config_path = os.path.join(LIB_PARENT, "IsolatedTCPythonLibrary", "utils", "config.json")
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)

    # Init ThreatConnect
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    try:
        tc = ThreatConnect(
            api_aid=api_access_id,
            api_sec=api_secret_key,
            api_org=api_default_org,
            api_url=api_base_url,
        )
    except TypeError:
        tc = ThreatConnect(
            access_id=api_access_id,
            secret_key=api_secret_key,
            default_org=api_default_org,
            api_url=api_base_url,
        )

    if hasattr(tc, "set_verify_ssl"):
        tc.set_verify_ssl(VERIFY_SSL)
    else:
        setattr(tc, "_verify_ssl", VERIFY_SSL)

    # RequestObject
    ro = RequestObject()
    ro.set_http_method("GET")
    ro.set_owner_allowed(True)

    # Build cutoff (UTC midnight)
    start_date = (datetime.now(pytz.UTC) - timedelta(days=days)).date()
    start = f"{start_date}T00:00:00Z"

    # IMPORTANT: verify exact typeName strings for your tenant
    type_names = [
        "Address", "Email Address", "File", "Host", "URL", "ASN", "CIDR",
        "Email Subject", "Hashtag", "Mutex", "Registry Key",
        "Stripped URL", "User Agent",
    ]
    type_name_condition = ", ".join([f'"{t}"' for t in type_names])

    # Query indicators (with pagination)
    final_results = []
    for owner in OWNERS:
        try:
            tql_raw = (
                f'ownerName EQ "{owner}" AND '
                f'typeName IN ({type_name_condition}) AND '
                f'lastObserved >= "{start}"'
            )
            tql_encoded = urllib.parse.quote(tql_raw)

            result_start = 0
            page_size = 10000

            while True:
                ro.set_http_method("GET")
                ro.set_request_uri(
                    f"/v3/indicators?tql={tql_encoded}"
                    f"&fields=tags,observations,associatedGroups,falsePositives"
                    f"&resultStart={result_start}&resultLimit={page_size}"
                )
                response = tc.api_request(ro)

                ctype = (response.headers.get("content-type") or "").lower()
                if "application/json" not in ctype:
                    break

                payload = response.json() or {}
                data = payload.get("data", [])
                if not data:
                    break

                final_results.append(payload)
                # TC v3 returns 'count' / 'resultCount'—use length to be safe
                if len(data) < page_size:
                    break
                result_start += page_size

        except Exception as e:
            print(f"Failed to query indicators for {owner}: {e}")

    # Normalize
    normalized_data = []
    for result in final_results:
        for item in result.get("data", []):
            if isinstance(item, dict) and "summary" in item:
                normalized_data.append(item)

    if normalized_data:
        df = pd.json_normalize(normalized_data)

        # Try to pick a stable "indicator" column.
        # If API provides a specific value key, prefer it; otherwise fallback to summary head token.
        indicator_col = None
        for cand in ["indicator", "value", "name", "summary"]:
            if cand in df.columns:
                indicator_col = cand
                break

        if indicator_col == "summary" or indicator_col is None:
            df["indicator"] = df["summary"].astype(str).str.split().str[0].str.strip()
        else:
            df["indicator"] = df[indicator_col].astype(str).str.strip()

        df.drop_duplicates(subset="indicator", inplace=True)

        if "lastObserved" in df.columns:
            df["lastObserved"] = pd.to_datetime(df["lastObserved"], utc=True, errors="coerce")
            df = df[df["lastObserved"] >= pd.to_datetime(start, utc=True)]

        observed_src = df.reset_index(drop=True)
    else:
        observed_src = pd.DataFrame()

    return observed_src


In [3]:
df = get_tc_data(days=7)
display(df)


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,lastFalsePositive,md5,url,indicator
0,5629499539130854,2025-04-25T13:02:39Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-15T15:30:02Z,4.0,55,Executive Summary: \n\nThe following TOR node...,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 6755399446014059, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.84.107.128
1,5629499539130853,2025-04-25T13:02:39Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-15T15:30:02Z,4.0,55,Executive Summary: \n\nThe following TOR node...,...,"[{'id': 38932, 'name': 'Active Scanning', 'des...","[{'id': 6755399446014059, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.192.3.74
2,6755399464324032,2025-07-21T17:28:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-15T15:25:37Z,5.0,93,HHS CSIRC observed an attempted exploitation. ...,...,"[{'id': 1470973, 'name': 'Linen Typhoon', 'las...","[{'id': 6755399454000275, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.236.230.76
3,5629499561026146,2025-07-25T18:44:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-15T15:25:19Z,3.0,73,INC9164255,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,117.50.216.193
4,6755399463603582,2025-07-17T15:18:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-15T15:18:18Z,4.0,76,NaN,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 6755399453001638, 'dateAdded': '2025-0...",dub.sh,False,False,NaN,NaN,NaN,NaN,dub.sh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1364,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85,NaN,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...","[{'id': 291847, 'dateAdded': '2023-12-15T13:16...",NaN,NaN,NaN,http://selligenttier.naylorcampaigns.com,NaN,NaN,selligenttier.naylorcampaigns.com,selligenttier.naylorcampaigns.com
1365,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,"[{'id': 34822, 'name': 'business email comprom...","[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,geo.netsupportsoftware.com/location/loca.asp
1366,4585199,2024-04-30T15:02:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-05-05T23:23:15Z,4.0,54,VA users received email messages from <Good Ch...,...,"[{'id': 39680, 'name': 'Initial Access: Phishi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,track.christianityreport.net,track.christianityreport.net
1367,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,https://google,NaN,NaN,google,google
